<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/08_value_function_DP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from functools import partial
from typing import Callable
from matplotlib import animation
from IPython.display import HTML


## Optimal Value & Policy
We compare candidate policies by their empirical value on a controlled mass–spring–damper system, and manually update the policy to improve it. The process of improving a policy is referred to as *policy iteration*.

\begin{aligned}
\dot{p} = v,\quad \dot{v} = -\omega^2 p -2\zeta \omega v+ u.
\end{aligned}

This is a manual policy iteration exercise:
1. Start with a policy.
2. Evaluate its value map $V_{\pi}(x)$ over states of interest.
3. Perturb the policy and see if it's better.

We use $ x=[\,p\; v\,]^\top $ (position, velocity) and $ L(x,u)=x^\top Q x + R u^2 $,
with $ Q=\mathrm{diag}(q_p, q_v) $, $ R>0 $.



Set up the dynamics and policy

In [ ]:
def msd_dynamics(
    state: jnp.ndarray, control: jnp.ndarray, dt: float, omega: float, zeta: float
):
    p, v = state
    p_next = p + dt * v
    v_next = v + dt * (-(omega**2) * p - 2.0 * zeta * omega * v + control)
    return jnp.array([p_next, v_next])


# [TODO]: in class
def make_linear_policy(kp: float, kv: float, u_max: float = None):
    def pi(x):
        # Extract position and velocity from state
        # Calculate control input using position and velocity gains

        if u_max is not None: # use jnp.clip to clip the control input to the max acceleration/braking
            u = ...
        return u
    # return the policy function pi
    return NotImplementedError("make_linear_policy function not implemented yet")

def stage_cost(x, u, Q, R):
    """
    L(x,u) = x^T Q x + u^T R u, with Q (6x6) and R (3x3 or scalar).
    """
    return x.T @ Q @ x +  R * u**2


# function to make the value rollout
def make_value_rollout(
    policy_fn: Callable[[jnp.ndarray], jnp.ndarray],
    dt: float,
    n_timesteps: int,
    omega: float,
    zeta: float,
    Q: jnp.ndarray,
    R: float,
):
    @jax.jit
    def rollout(x0): # x0 is the initial state
        def step(x, _): # x is the current state, _ is the unused input
            u = policy_fn(x) # get the control input from the policy function
            c = stage_cost(x, u, Q, R) # get the stage cost
            x_next = msd_dynamics(x, u, dt, omega, zeta)
            return x_next, (x, u, c) # return the next state and the cost

        _, (x_hist, u_hist, c_hist) = jax.lax.scan(step, x0, jnp.zeros(n_timesteps)) # scan the step function over the initial state and the number of timesteps
        total_cost = jnp.sum(c_hist) * dt # sum the costs and multiply by the timestep to get the total cost
        return total_cost, (x_hist, u_hist)

    return rollout

# function to make the value map
def value_map(policy_fn, Q, R, dt, N, omega, zeta, p_grid, v_grid):
    rollout = make_value_rollout(policy_fn, dt, N, omega, zeta, Q, R) # make the value rollout function

    def V_from_state(x0):
        V, _ = rollout(x0) # get the value from the rollout function
        return V

    PP, VV = jnp.meshgrid(p_grid, v_grid, indexing="xy") # create a meshgrid of the position and velocity
    X0 = jnp.stack([PP, VV], axis=-1)

    V_vmapped = jax.vmap(jax.vmap(V_from_state, in_axes=0), in_axes=0) # map the value from the state function over the meshgrid
    V_vals = V_vmapped(X0)
    return PP, VV, V_vals

In [ ]:
# Physical parameters
omega = 1.2  # natural frequency
zeta = 0.15  # damping ratio

# Cost weights
q_p, q_v, r = 1.0, 0.2, 0.05
Q = jnp.diag(jnp.array([q_p, q_v]))
R = r

# Discretization/horizon
dt = 0.05
N = 240


## (a) Choose policies
### (i) Select two sets of gains to define two policies.

Pick two linear policies $u=-(k_p\,p+k_v\,v)$ with **stabilizing** gains (e.g., $k_p>0, k_v>0$).



[TODO]

Describe qualitatively how you expect each of your policies to behave.

[student response here]

In [ ]:
# Define your two policies here
# Choose a value for u_max. You can set it to None for no saturation.
pi_1 = make_linear_policy(kp=0.0, kv=1.1, u_max=15.0)  # change me
pi_2 = make_linear_policy(kp=1.2, kv=0.0, u_max=15.0)  # change me

Now, let's plot out the value function of each of the policies

In [ ]:
# plotting out the value functions

p_min, p_max, p_pts = -20.0, 20.0, 121
v_min, v_max, v_pts = -20.0, 20.0, 121

# create a grid of the position and velocity
p_grid = jnp.linspace(p_min, p_max, p_pts)
v_grid = jnp.linspace(v_min, v_max, v_pts)

extent = [p_min, p_max, v_min, v_max]

# Compute value maps and dominance
PP, VV, V1 = value_map(pi_1, Q, R, dt, N, omega, zeta, p_grid, v_grid)
_, _, V2 = value_map(pi_2, Q, R, dt, N, omega, zeta, p_grid, v_grid)

plt.figure(figsize=(7, 3))

plt.subplot(1, 2, 1)
plt.title(r"$V_{\pi_1}(p,v)$")
plt.imshow(V1, origin="lower", extent=extent, aspect="auto")
plt.colorbar()
plt.contour(PP, VV, V1, linewidths=0.5, colors="black")
plt.xlabel("position p")
plt.ylabel("velocity v")
plt.axis("equal")


plt.subplot(1, 2, 2)
plt.title(r"$V_{\pi_2}(p,v)$")
plt.imshow(V2, origin="lower", extent=extent, aspect="auto")
plt.colorbar()
plt.contour(PP, VV, V2, linewidths=0.5, colors="black")
plt.xlabel("position p")
plt.ylabel("velocity v")
plt.axis("equal")


plt.tight_layout()
plt.show()


### (ii) See which one dominates.
Does this align with your understanding for your choice of policies.

In [ ]:
# see which one is better at each state
dominates_1 = V1 < V2 # True if V1 is less than V2 at that state
dominates_2 = ~dominates_1 # True if V2 is greater than V1 at that state


plt.figure(figsize=(7, 3))

plt.subplot(1, 2, 1)
plt.title(r"$V_{\pi_2}(p,v) > V_{\pi_1}(p,v)$")
plt.imshow(dominates_2 * 1, origin="lower", extent=extent, aspect="auto")
plt.colorbar()
plt.xlabel("position p")
plt.ylabel("velocity v")
plt.axis("equal")


plt.subplot(1, 2, 2)
plt.title(r"$V_{\pi_2}(p,v) - V_{\pi_1}(p,v)$")
plt.imshow(V2 - V1, origin="lower", extent=extent, aspect="auto")
plt.colorbar()
plt.contour(PP, VV, V2 - V1, linewidths=0.5, colors="black")
plt.xlabel("position p")
plt.ylabel("velocity v")
plt.axis("equal")



plt.tight_layout()
plt.show()

## (b) Policy improvement
Now, pick the better policy $\tilde{\pi}$ (describe the metric you use to determine which one is better).
Add some perturbations to the gains of that policy and see which set of perturbations would provide the greatest improvement.

[TODO]: in peers/hw

Briefly describe qualitatively the ways in which your new policy does better. Does this align with your intuition?

[student response here]

In [ ]:
V_best = V1 # or V2 # UPDATE ME!
pi_tilde = make_linear_policy(kp=1.2, kv=0.9, u_max=15.0) # UPDATE ME!


In [ ]:
# plot out the value function for the perturbed policy
# compute value function for the perturbed policy
_, _, V_tilde = value_map(pi_tilde, Q, R, dt, N, omega, zeta, p_grid, v_grid)
# compute the dominance of the perturbed policy vs the original policy
dom_tilde_vs_pi1 = V_tilde < V1



plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.title(r"$V_{\tilde{\pi}}(p,v)$")
plt.imshow(
    jnp.array(V_tilde),
    origin="lower",
    extent=[p_min, p_max, v_min, v_max],
    aspect="auto",
)
plt.xlabel("p")
plt.ylabel("v")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.title(r"Dominance: $\tilde{\pi}$ vs $\pi_1$")
plt.imshow(
    jnp.array(dom_tilde_vs_pi1).astype(float),
    origin="lower",
    extent=[p_min, p_max, v_min, v_max],
    aspect="auto",
)
plt.xlabel("p")
plt.ylabel("v")
cb = plt.colorbar()
cb.set_ticks([0, 1])
cb.set_ticklabels([r"$\pi_1$", r"$\tilde{\pi}$"])
plt.tight_layout()
plt.show()
